# **TaniMol: 01 - Preprocessing Pipeline**

This notebook loads raw bioactivity data from the ChEMBL database for five DNA repair protein inhibitor targets, cleans and standardizes the molecules, and produces a processed dataset ready for fingerprinting and similarity analysis.

**Targets:** PARP1, PARP2, ATR, ATM, DNA-PKcs  
**Source:** ChEMBL v36 (local SQLite database)  
**Output:** `data/processed/cleaned_activities.csv`

In [1]:
from src.config import DB_PATH, TARGETS, MIN_CONFIDENCE, ACTIVITY_TYPES, ACTIVITY_UNITS, OUTPUT_PATH
from src.preprocessing import (
    fetch_activity_data,
    drop_missing_values,
    validate_smiles,
    standardize_molecules,
    deduplicate,
    compute_pic50,
    save_cleaned_data,
)

from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")

### **1. Data Acquisition**

Query the local ChEMBL SQLite database for bioactivity records (IC50, Ki) across all configured targets. Only records with confidence score ≥ 7 and units in nM are included.

In [2]:
df = fetch_activity_data(DB_PATH, TARGETS, MIN_CONFIDENCE, ACTIVITY_TYPES, ACTIVITY_UNITS)
print(f"Downloaded {len(df)} rows")
df.head()

Downloaded 4153 rows


,molecule_chembl_id,canonical_smiles,target_chembl_id,standard_type,standard_relation,standard_value,standard_units,pchembl_value
0,CHEMBL108702,Nc1cccc(-c2ccc(C(=O)CNC(=O)CCn3c4c(c(=O)[nH]c3...,CHEMBL3105,IC50,=,20.0,nM,7.70
1,CHEMBL418816,O=C(CCn1c2c(c(=O)[nH]c1=O)CCC2)NCC(=O)N1CCN(c2...,CHEMBL3105,IC50,=,3000.0,nM,5.52
2,CHEMBL108968,O=C(CCn1c2c(c(=O)[nH]c1=O)CSCC2)NCC(=O)c1ccc(O...,CHEMBL3105,IC50,=,30.0,nM,7.52
3,CHEMBL430707,O=C(CCn1c2c(c(=O)[nH]c1=O)CSCC2)NCC(=O)N1CCN(c...,CHEMBL3105,IC50,=,35.0,nM,7.46
4,CHEMBL321638,COc1ccc(-c2ccc(C(=O)CNC(=O)CCn3c4c(c(=O)[nH]c3...,CHEMBL3105,IC50,=,15.0,nM,7.82


### **2. Drop Missing Values**

Remove rows where `standard_value` (IC50/Ki measurement) is missing — these rows cannot be used for activity analysis.

In [3]:
rows_before = len(df)
df = drop_missing_values(df)
print(f"Dropped {rows_before - len(df)} rows with missing standard_value.")

Dropped 0 rows with missing standard_value.


### **3. SMILES Validation**

Parse each SMILES string with RDKit and remove any that cannot be interpreted as a valid molecule. ChEMBL data is well-curated, so we expect very few (if any) invalid entries.

In [4]:
rows_before = len(df)
df = validate_smiles(df)
print(f"Dropped {rows_before - len(df)} rows with invalid SMILES.")

Dropped 0 rows with invalid SMILES.


### **4. Molecule Standardization**

Standardize molecular representations to ensure consistent comparisons:
- **Salt stripping** — remove counter-ions, keep only the largest (active) fragment
- **Charge neutralization** — convert ionized forms to neutral
- **Tautomer canonicalization** — choose a single canonical tautomeric form

This step does not remove rows — it modifies the SMILES in place.

In [5]:
smiles_before = df["canonical_smiles"].copy()

df = standardize_molecules(df)

changed = (smiles_before != df["canonical_smiles"]).sum()
print(f"Standardized {changed} out of {len(df)} SMILES.")

Standardizing: 100%|██████████| 4153/4153 [00:53<00:00, 78.00it/s] 

Standardized 1113 out of 4153 SMILES.


### **5. Deduplication**

After standardization, some molecules that had different SMILES representations now match. We group by `(target, SMILES)` and keep only the row with the lowest (best) IC50 value.

In [6]:
rows_before = len(df)
df = deduplicate(df)
print(f"Dropped {rows_before - len(df)} duplicate rows.")

Dropped 809 duplicate rows.


### **6. Compute pIC50**

Fill in missing `pchembl_value` entries by computing pIC50 = −log₁₀(IC50 × 10⁻⁹). Rows where the calculation is impossible (IC50 ≤ 0) are removed.

In [7]:
missing_before = df["pchembl_value"].isna().sum()

df = compute_pic50(df)

missing_after = df["pchembl_value"].isna().sum()
print(f"Filled {missing_before - missing_after} missing pchembl values.")

rows_before = len(df)
df = df.dropna(subset=["pchembl_value"])
print(f"Dropped {rows_before - len(df)} rows with invalid IC50 (≤ 0).")

Computing pIC50: 100%|██████████| 3344/3344 [00:00<00:00, 51689.55it/s]

Filled 21 missing pchembl values.
Dropped 0 rows with invalid IC50 (≤ 0).


### **7. Save Processed Data**

Export the cleaned dataset to CSV with a per-target row count summary.

In [8]:
save_cleaned_data(df, OUTPUT_PATH, targets=TARGETS)

Saved 3344 rows to /home/stanuch/Dev/TaniMol/data/processed/cleaned_activities.csv

Rows per target:
- PARP1 (CHEMBL3105): 3344
